# 📈 MODEL 2: Dự báo Suy giảm Chuỗi Thời gian 5 Phút Tương lai bằng XGBoost

---
### 📌 Cơ chế Tự Chọn Đặc trưng từ Unified Dataset (`aiops_network_dataset.csv`):
- **File Dataset dùng chung cho cả 3 Model:** `aiops_network_dataset.csv` (`700 mẫu × 117 cột`).
- **Tập đặc trưng Model 2 tự chọn (`84 đặc trưng cửa sổ trượt Sliding Window`):** Gồm 14 KPIs tức thời + 70 đặc trưng biến thiên theo thời gian (`*_roll_mean`, `*_roll_std`, `*_delta1`, `*_trend_slope`, `*_sustained_ratio`).
- **Cột Nhãn Mục tiêu (`Target`) Model 2 tự chọn:** `degraded_forecast_5m` (`0`: 5 phút tới ổn định/tự khỏi, `1`: 5 phút tới vẫn suy giảm).
- **File Model xuất ra:** `model2_xgboost_forecaster.pkl`

In [ ]:
import glob
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (11, 6)

## 1. Đọc Unified Dataset (`aiops_network_dataset.csv`) & Tự Chọn 84 Đặc trưng Chuỗi thời gian

In [ ]:
candidates = (
    glob.glob('/kaggle/input/**/aiops_network_dataset.csv', recursive=True)
    + ['../../dataset/aiops_network_dataset.csv', '../dataset/aiops_network_dataset.csv', 'aiops_network_dataset.csv']
)
csv_path = next((p for p in candidates if os.path.exists(p)), None)
assert csv_path is not None, 'Hãy upload file aiops_network_dataset.csv lên Kaggle!'

df = pd.read_csv(csv_path).replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Model 2 tự động chọn 14 KPIs cốt lõi + 70 đặc trưng cửa sổ trượt (tổng 84 features)
KEY_TIMESERIES_KPIS = [
    'wan1_rx_bytes_rate', 'wan1_tx_bytes_rate', 'branch_tcp_retrans_rate',
    'wan_tcp_syn_retrans_rate', 'wan_tcp_timeouts_rate', 'fw_rx_pkts_rate',
    'fw_avg_pkt_size', 'client_lan_tx_pkts_rate', 'srv_tcp_out_rsts_rate',
    'srv_tcp_passive_opens_rate', 'srv_cpu_usage', 'nginx_req_rate',
    'api_req_rate', 'api_latency_avg'
]
ROLLING_SUFFIXES = ('_roll_mean', '_roll_std', '_delta1', '_trend_slope', '_sustained_ratio')
rolling_cols = [c for c in df.columns if c.endswith(ROLLING_SUFFIXES)]

MODEL2_FEATURES = KEY_TIMESERIES_KPIS + rolling_cols
TARGET_COL = 'degraded_forecast_5m'

print(f'✅ Đã nạp Unified Dataset: {csv_path} (Tổng {df.shape[0]} mẫu x {df.shape[1]} cột)')
print(f'📈 Model 2 tự động chọn : {len(MODEL2_FEATURES)} đặc trưng Sliding Window | Target: {TARGET_COL}')
display(df[['datetime', *MODEL2_FEATURES[:6], TARGET_COL]].head())

## 2. Huấn luyện & Đối sánh XGBoost Forecaster với các Mô hình Chuỗi thời gian

In [ ]:
X = df[MODEL2_FEATURES].values
y = df[TARGET_COL].values.astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

scaler = RobustScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)
X_all_s = RobustScaler().fit_transform(X)

candidates = {
    'XGBoost_Forecaster': XGBClassifier(
        n_estimators=250, learning_rate=0.07, max_depth=5, subsample=0.9, colsample_bytree=0.9,
        reg_alpha=0.1, reg_lambda=1.0, eval_metric='logloss', random_state=42, n_jobs=-1
    ),
    'RandomForest_Forecaster': RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1),
    'ExtraTrees_Forecaster': ExtraTreesClassifier(n_estimators=300, max_depth=14, class_weight='balanced', random_state=42, n_jobs=-1),
    'GradientBoosting_Forecaster': GradientBoostingClassifier(n_estimators=200, learning_rate=0.08, max_depth=4, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
trained_models = {}
plt.figure(figsize=(8, 6))

for name, clf in candidates.items():
    clf.fit(X_tr_s, y_tr)
    preds = clf.predict(X_te_s)
    probs = clf.predict_proba(X_te_s)[:, 1]
    cv_scores = cross_val_score(clf, X_all_s, y, cv=cv, scoring='f1', n_jobs=-1)
    trained_models[name] = (clf, preds, probs)
    
    auc = roc_auc_score(y_te, probs)
    fpr, tpr, _ = roc_curve(y_te, probs)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.4f})')
    
    results.append({
        'Model 2 Algorithm': name,
        'Accuracy': accuracy_score(y_te, preds),
        'Precision': precision_score(y_te, preds),
        'Recall': recall_score(y_te, preds),
        'F1-Score': f1_score(y_te, preds),
        'ROC-AUC': auc,
        '5-Fold CV F1': np.mean(cv_scores),
    })

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.title('ROC Curve - Dự báo Suy giảm Chuỗi thời gian 5 Phút Tương lai (Model 2: XGBoost)', fontweight='bold')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.legend(loc='lower right')
plt.show()

res_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False)
display(res_df.round(4))

In [ ]:
xgb_clf, xgb_preds, xgb_probs = trained_models['XGBoost_Forecaster']
print('🏆 Báo cáo Phân loại Chi tiết của XGBoost_Forecaster (Model 2):')
print(classification_report(y_te, xgb_preds, target_names=['5m Ổn định / Tự khỏi (0)', '5m Suy giảm Kéo dài (1)'], digits=4))

cm = confusion_matrix(y_te, xgb_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=['Dự báo Ổn định (0)', 'Dự báo Suy giảm (1)'], yticklabels=['Thực tế Ổn định (0)', 'Thực tế Suy giảm (1)'])
plt.title('Confusion Matrix - Model 2 (XGBoost 5-Min Forecaster)', fontweight='bold')
plt.tight_layout()
plt.show()

imp_df = pd.DataFrame({'Time-Series Feature': MODEL2_FEATURES, 'Importance': xgb_clf.feature_importances_}).sort_values('Importance', ascending=False).head(15)
plt.figure(figsize=(11, 6))
sns.barplot(x='Importance', y='Time-Series Feature', data=imp_df, palette='rocket')
plt.title('Top 15 Đặc trưng Chuỗi thời gian Quyết định Dự báo 5 Phút (Model 2: XGBoost)', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Đóng gói & Xuất File `model2_xgboost_forecaster.pkl`

In [ ]:
final_scaler = RobustScaler()
X_all_scaled = final_scaler.fit_transform(X)

final_xgb = XGBClassifier(
    n_estimators=250, learning_rate=0.07, max_depth=5, subsample=0.9, colsample_bytree=0.9,
    reg_alpha=0.1, reg_lambda=1.0, eval_metric='logloss', random_state=42, n_jobs=-1
)
final_xgb.fit(X_all_scaled, y)

artifact_m2 = {
    'model_type': 'XGBoost_5mForecaster',
    'model': final_xgb,
    'scaler': final_scaler,
    'feature_names': MODEL2_FEATURES,
    'window_size': 5,
    'target_name': TARGET_COL,
}

joblib.dump(artifact_m2, 'model2_xgboost_forecaster.pkl')
print('✅ Đã xuất thành công Model 2: model2_xgboost_forecaster.pkl')